# Postprocessing: the polarization from finished DFT runs

This tutorial computes the change in polarization between a polar and a
non-polar structure from their DFT runs, for BaTiO₃ in both Quantum ESPRESSO
and VASP. It then shows the same calculation for spin-polarized runs and runs
on the irreducible Brillouin zone (IBZ), and a chain of interpolated images for
LiNbO₃.

**Data.** Wavefunction files are too large to keep in the repository. The
notebook reads the runs from two folders, given by environment variables (the
same ones the test suite uses):

| | |
|---|---|
| `BFD_BATIO3_SPIN_RUNS` | `batio3_spin_ibz_runs`: BaTiO₃ in QE and VASP, spin and non-spin, full BZ and IBZ |
| `BFD_LINBO3_RUNS` | `linbo3_ibz_interp_runs`: LiNbO₃ with one interpolated image |

Point them at your own runs to use your own data.

In [1]:
import os
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="IProgress not found")  # tqdm, imported by pymatgen

import berry_flux_diag as bfd
from berry_flux_diag.Overlaps import chain_polarization

bfd.configure_logging("WARNING");  # "INFO" shows each step and the ionic and electronic parts

batio3_runs = Path(os.environ["BFD_BATIO3_SPIN_RUNS"])
linbo3_runs = Path(os.environ["BFD_LINBO3_RUNS"]) / "runs"

## VASP

For each structure the parser needs the `POSCAR`, `WAVECAR` and `vasprun.xml`
(from the run directory), and one `POTCAR`. `Overlaps.compute_polarization`
returns the norm of the polarization change in µC/cm², and a dictionary of
diagnostics.

In [2]:
pol_dir = batio3_runs / "VASP/nospin_fbz/pol"   # polar structure's run
np_dir = batio3_runs / "VASP/nospin_fbz/np"     # non-polar structure's run

vasp_parse_dict = bfd.VASPParser.vasp_parser(
    pol_dir / "POSCAR", np_dir / "POSCAR",
    pol_dir / "WAVECAR", np_dir / "WAVECAR",
    pol_dir / "POTCAR",
    pol_dir, np_dir,   # directories holding each run's vasprun.xml
)

vasp_ovlps = bfd.Overlaps.Overlaps(vasp_parse_dict)
vasp_pol, vasp_debug = vasp_ovlps.compute_polarization()
print(f"VASP: {vasp_pol:.5f} uC/cm^2")

VASP: 45.78751 uC/cm^2


## Quantum ESPRESSO

QE needs each structure's `.xml` file and `.save/` folder (with the
wavefunctions in HDF5 format), and a `pw.x` output file (for the valence
charges).

In [3]:
pol_calc = batio3_runs / "QE/nospin_fbz/pol/calculations"
np_calc = batio3_runs / "QE/nospin_fbz/np/calculations"

qe_parse_dict = bfd.QEParser.qe_parser(
    str(pol_calc / "batio3_pol.xml"), str(np_calc / "batio3_np.xml"),
    str(pol_calc / "batio3_pol.save") + "/", str(np_calc / "batio3_np.save") + "/",
    str(batio3_runs / "QE/nospin_fbz/np/batio3_np.out"),
)

qe_ovlps = bfd.Overlaps.Overlaps(qe_parse_dict)
qe_pol, qe_debug = qe_ovlps.compute_polarization()
print(f"QE:   {qe_pol:.5f} uC/cm^2")

QE:   45.50666 uC/cm^2


The two codes differ by about 0.3 µC/cm², because they use different
pseudopotentials and k-point meshes (VASP 6×6×6, QE 5×5×5).

In [4]:
del vasp_parse_dict, vasp_ovlps, qe_parse_dict, qe_ovlps  # free the wavefunctions

## Spin-polarized runs, and runs on the irreducible Brillouin zone

Nothing changes in the calls:
- **Spin-polarized (collinear) runs** are recognized from the DFT output. The
  two spin channels' Berry phases are added.
- **IBZ runs** are recognized from the k-point list (one that is not a full
  mesh). The states are unfolded to the full mesh before anything is
  computed. Use the NSCF step's output: `nscf/` in VASP, and the overwritten
  `.save` folder in QE.

BaTiO₃ is non-magnetic, so its spin runs have zero moment and should match
the non-spin runs. The IBZ runs should match the full-BZ runs. In both cases
the agreement should be about 10⁻³ µC/cm², the DFT convergence.

In [5]:
def batio3_pair(code, case):
    d = batio3_runs / code / case
    if code == "VASP":
        sub = "nscf" if case.endswith("_ibz") else ""
        p, n = d / "pol" / sub, d / "np" / sub
        return bfd.VASPParser.vasp_parser(p / "POSCAR", n / "POSCAR", p / "WAVECAR", n / "WAVECAR",
                                          p / "POTCAR", p, n)
    out = "batio3_np_nscf.out" if case.endswith("_ibz") else "batio3_np.out"
    return bfd.QEParser.qe_parser(
        str(d / "pol/calculations/batio3_pol.xml"), str(d / "np/calculations/batio3_np.xml"),
        str(d / "pol/calculations/batio3_pol.save") + "/", str(d / "np/calculations/batio3_np.save") + "/",
        str(d / "np" / out))

cases = ["nospin_fbz", "nospin_ibz", "spin_fbz", "spin_ibz"]
results = {}
for code in ("QE", "VASP"):
    for case in cases:
        parse = batio3_pair(code, case)
        results[code, case] = float(bfd.Overlaps.Overlaps(parse).compute_polarization()[0])
        print(f"{code:4} {case:10}  spin: {parse['spin_pol']!s:5}  "
              f"k-points after unfolding: {len(parse['kpoint_list']):3}  P = {results[code, case]:.5f} uC/cm^2")
        del parse

QE   nospin_fbz  spin: False  k-points after unfolding: 125  P = 45.50666 uC/cm^2


QE   nospin_ibz  spin: False  k-points after unfolding: 125  P = 45.50895 uC/cm^2


QE   spin_fbz    spin: True   k-points after unfolding: 125  P = 45.50574 uC/cm^2


QE   spin_ibz    spin: True   k-points after unfolding: 125  P = 45.50774 uC/cm^2


VASP nospin_fbz  spin: False  k-points after unfolding: 216  P = 45.78751 uC/cm^2


VASP nospin_ibz  spin: False  k-points after unfolding: 216  P = 45.78986 uC/cm^2


VASP spin_fbz    spin: True   k-points after unfolding: 216  P = 45.78727 uC/cm^2


VASP spin_ibz    spin: True   k-points after unfolding: 216  P = 45.78895 uC/cm^2


In [6]:
for code in ("QE", "VASP"):
    spread = max(results[code, c] for c in cases) - min(results[code, c] for c in cases)
    print(f"{code}: largest difference between the four cases = {spread:.1e} uC/cm^2")

QE: largest difference between the four cases = 3.2e-03 uC/cm^2
VASP: largest difference between the four cases = 2.6e-03 uC/cm^2


## A chain of interpolated images: LiNbO₃

When the polar and non-polar structures are too far apart, preprocessing adds
interpolated images (see the preprocessing tutorial), and each image gets its
own DFT run. `chain_polarization` takes the parsed adjacent pairs in order,
(pol, image 1), …, (image n, np). It adds their electronic contributions,
and computes the ionic contribution once, from the two end points.

Pass a generator, as here, so that only one parsed pair is held in memory at a
time.

LiNbO₃ was interpolated with one image: `pol_orig` → `interp_1` → `np_trans`,
0.19 Å per step.

In [7]:
tags = ["pol_orig", "interp_1", "np_trans"]

def linbo3_pair(a, b):
    p, n = linbo3_runs / "VASP/nospin_fbz" / a, linbo3_runs / "VASP/nospin_fbz" / b
    return bfd.VASPParser.vasp_parser(p / "POSCAR", n / "POSCAR", p / "WAVECAR", n / "WAVECAR",
                                      p / "POTCAR", p, n)

result = chain_polarization(linbo3_pair(a, b) for a, b in zip(tags[:-1], tags[1:]))

print(f"chain:  {result['polarization_norm']:.5f} uC/cm^2")
print("P (Cartesian, uC/cm^2):", result["polarization_cartesian"].round(4))
for (a, b), elec in zip(zip(tags[:-1], tags[1:]), result["elec_change_per_pair"]):
    print(f"  electronic change {a} -> {b}:", [round(float(x), 5) for x in elec])

chain:  84.29088 uC/cm^2
P (Cartesian, uC/cm^2): [-8.02735e+01 -2.32000e-02 -2.57123e+01]
  electronic change pol_orig -> interp_1: [-0.44259, -0.7631, -0.7631]
  electronic change interp_1 -> np_trans: [-0.4774, -0.79751, -0.79751]


Here the two end points are only 0.38 Å apart, so the direct pair is on the
right branch as well, and should agree with the chain to rounding error:

In [8]:
direct = chain_polarization([linbo3_pair("pol_orig", "np_trans")])
print(f"direct: {direct['polarization_norm']:.5f} uC/cm^2")
print(f"chain - direct: {result['polarization_norm'] - direct['polarization_norm']:.1e} uC/cm^2")

direct: 84.29088 uC/cm^2
chain - direct: -1.1e-13 uC/cm^2


For structures further apart, the direct pair can land a whole polarization
quantum off, and the chain finds the right branch. The test
`tests/test_chain_polarization.py` shows this for LiNbO₃ structures 0.79 Å
apart.